In [ ]:
# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier



In [1]:

# STEP 2: LOAD THE DATASET
# Replace with your actual file path
df = pd.read_csv("creditcard.csv")

print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

    

In [3]:
# STEP 3: CHECK CLASS DISTRIBUTION
print("\nClass Distribution:")
print(df['Class'].value_counts())

print("\nClass Distribution (%):")
print(df['Class'].value_counts(normalize=True) * 100)


# ==========================================
# STEP 4: CHECK FOR MISSING VALUES
# ==========================================
print("\nMissing Values:")
print(df.isnull().sum().sum())


# ==========================================
# STEP 5: SEPARATE FEATURES (X) AND TARGET (y)
# ==========================================
X = df.drop('Class', axis=1)
y = df['Class']

print("\nFeature matrix shape:", X.shape)
print("Target vector shape:", y.shape)


# ==========================================
# STEP 6: TRAIN-TEST SPLIT
# ==========================================
# stratify=y preserves the fraud/non-fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())


Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Class Distribution (%):
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

Missing Values:
0

Feature matrix shape: (284807, 30)
Target vector shape: (284807,)

Training set shape: (227845, 30)
Test set shape: (56962, 30)

Training target distribution:
Class
0    227451
1       394
Name: count, dtype: int64

Test target distribution:
Class
0    56864
1       98
Name: count, dtype: int64


In [5]:
# STEP 7: SCALE FEATURES

# Important for Logistic Regression
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

print("\nScaled training data shape:", X_train_scaled.shape)
print("Scaled test data shape:", X_test_scaled.shape)


# STEP 8: OPTIONAL - CONVERT BACK TO DATAFRAME
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("\nPreprocessing complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)


Scaled training data shape: (227845, 30)
Scaled test data shape: (56962, 30)

Preprocessing complete.
X_train_scaled shape: (227845, 30)
X_test_scaled shape: (56962, 30)


In [6]:
# STEP 9: TRAIN AND EVALUATE MODELS : TOOK 2 minutes

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Define Models
models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42,
        max_depth=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

# Train and Evaluate
results = []

for name, model in models.items():

    print("\n" + "=" * 60)
    print(f"MODEL: {name}")
    print("=" * 60)

    # Logistic Regression benefits from scaled data.
    # Tree-based models can also use the scaled data here
    # for simplicity and consistent code.
    model.fit(X_train_scaled, y_train)

    # Predicted labels
    y_pred = model.predict(X_test_scaled)

    # Predicted probabilities for PR-AUC
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    pr_auc = average_precision_score(y_test, y_prob)

    # Store results
    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "PR-AUC": pr_auc
    })

    # Print confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Detailed report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Summary metrics
    print("\nSummary Metrics:")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")



MODEL: Logistic Regression

Confusion Matrix:
[[55478  1386]
 [    8    90]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9999    0.9756    0.9876     56864
           1     0.0610    0.9184    0.1144        98

    accuracy                         0.9755     56962
   macro avg     0.5304    0.9470    0.5510     56962
weighted avg     0.9982    0.9755    0.9861     56962


Summary Metrics:
Precision : 0.0610
Recall    : 0.9184
F1 Score  : 0.1144
PR-AUC    : 0.7190

MODEL: Decision Tree

Confusion Matrix:
[[55135  1729]
 [   12    86]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9696    0.9845     56864
           1     0.0474    0.8776    0.0899        98

    accuracy                         0.9694     56962
   macro avg     0.5236    0.9236    0.5372     56962
weighted avg     0.9981    0.9694    0.9829     56962


Summary Metrics:
Precision : 0.0474
Recall    : 0.877

In [7]:
# Compare All Models
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="PR-AUC",
    ascending=False
)

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(results_df)


MODEL COMPARISON
                 Model  Precision    Recall  F1 Score    PR-AUC
2        Random Forest   0.960526  0.744898  0.839080  0.853946
0  Logistic Regression   0.060976  0.918367  0.114358  0.718971
1        Decision Tree   0.047383  0.877551  0.089911  0.449784


# Conclusion: 
### The best model is the RandomForestClassifier.


### Interpretation
**Precision = 0.961**:  When Random Forest predicts fraud, it is correct about 96% of the time.

This means very few false alarms.


**Recall = 0.745**: It catches about 74.5% of all actual fraud cases.


**F1 Score = 0.839**: Best balance between precision and recall.


**PR-AUC = 0.854**: The highest overall performance on this imbalanced dataset.

Since fraud detection is highly imbalanced, PR-AUC is one of the most informative metrics.

---
### Why Logistic Regression Is Not Ideal

Recall is high (0.918), but precision is only 0.061.

That means only about 6% of flagged transactions are truly fraudulent, creating many false positives.

---
### Business Perspective

Suppose the model flags 100 transactions as fraud:

- Logistic Regression: ~6 are real fraud.
- Decision Tree: ~5 are real fraud.
- Random Forest: ~96 are real fraud.

This makes Random Forest much more practical for fraud investigation teams.

# (OPTIONAL: Took 30 minutes on CPU)
# Lets try to improve the random forest model through hyperparameter tuning 

In [16]:
# ==========================================
# HYPERPARAMETER TUNING FOR RANDOM FOREST
# ==========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import average_precision_score, make_scorer

# ------------------------------------------
# Base Model
# ------------------------------------------
rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------
# Parameter Search Space
# ------------------------------------------
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# ------------------------------------------
# Scoring Metric
# average_precision = PR-AUC
# ------------------------------------------
pr_auc_scorer = make_scorer(
    average_precision_score,
    response_method='predict_proba'
)

# ------------------------------------------
# Random Search
# n_iter=10 keeps it reasonably fast
# cv=3 is enough for demonstration
# ------------------------------------------
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10,
    scoring=pr_auc_scorer,
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------
# Fit Search
# ------------------------------------------
random_search.fit(X_train_scaled, y_train)

# ------------------------------------------
# Best Parameters
# ------------------------------------------
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validated PR-AUC:")
print(random_search.best_score_)

# ------------------------------------------
# Best Model
# ------------------------------------------
best_rf = random_search.best_estimator_

# ------------------------------------------
# Evaluate on Test Set
# ------------------------------------------
y_prob = best_rf.predict_proba(X_test_scaled)[:, 1]
y_pred = best_rf.predict(X_test_scaled)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)

print("\n" + "=" * 60)
print("TUNED RANDOM FOREST RESULTS")
print("=" * 60)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}

Best Cross-Validated PR-AUC:
0.8420975193411985

TUNED RANDOM FOREST RESULTS
Precision : 0.9610
Recall    : 0.7551
F1 Score  : 0.8457
PR-AUC    : 0.8590


# Conclusion: This is a good model

## Why This Is a Good Result
**Precision = 0.9610**:  When the model flags fraud, it is correct about 96.1% of the time.

**Recall = 0.7551**: The model catches about 75.5% of all fraud cases.

**F1 Score = 0.8457**: The best balance we have achieved so far.

**PR-AUC = 0.8590**: Our highest overall score.

# OPTIONAL
Note to self: I tried xgbost, lightgbm and they did not gave good result. Maybe try them again  with different hyperparameters

In [13]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 3.1/100.2 MB 14.2 MB/s eta 0:00:07
   --- ------------------------------------ 9.7/100.2 MB 22.4 MB/s eta 0:00:05
   ------ --------------------------------- 16.3/100.2 MB 25.6 MB/s eta 0:00:04
   --------- ------------------------------ 23.3/100.2 MB 27.3 MB/s eta 0:00:03
   ------------ --------------------------- 30.4/100.2 MB 28.4 MB/s eta 0:00:03
   -------------- ------------------------- 37.2/100.2 MB 28.9 MB/s eta 0:00:03
   ----------------- ---------------------- 43.0/100.2 MB 28.5 MB/s eta 0:00:03
   ------------------ --------------------- 47.2/100.2 MB 27.6 MB/s eta 0:00:02
   ------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# pip install catboost

#
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='PRAUC',
    auto_class_weights='Balanced',
    verbose=0,
    random_seed=42
)

cat_model.fit(X_train, y_train)

y_prob = cat_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [15]:
# ==========================================
# EVALUATE CATBOOST MODEL
# ==========================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Predicted probabilities
y_prob = cat_model.predict_proba(X_test)[:, 1]

# Predicted labels (threshold = 0.5)
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)

# Print results
print("\n" + "=" * 60)
print("MODEL: CATBOOST")
print("=" * 60)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nSummary Metrics:")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")


MODEL: CATBOOST

Confusion Matrix:
[[56829    35]
 [   13    85]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9994    0.9996     56864
           1     0.7083    0.8673    0.7798        98

    accuracy                         0.9992     56962
   macro avg     0.8541    0.9334    0.8897     56962
weighted avg     0.9993    0.9992    0.9992     56962


Summary Metrics:
Precision : 0.7083
Recall    : 0.8673
F1 Score  : 0.7798
PR-AUC    : 0.8504


# Conlcusion: 
Random forest is better